# Parte 3

O objetivo desta parte do trabalho é estudar o comportamento dos otimizadores de consulta dos SGBDs através do exame e análise dos planos de execução para consultas SQL sobre tabelas que serão fornecidos. Será bastante utilizado o comando `EXPLAIN ANALYZE`, que permite visualizar todas as etapas envolvidas no processamento de uma consulta. Usaremos para isso a tabela [movies](https://drive.google.com/file/d/1W6wovSsVu4B0OIo_tsSBBHi8WRKQqnat/view)

Configuração Inicial:

In [19]:
# Conectar ao banco de dados PostgreSQL
import psycopg2

config = {
    'dbname': 'icomp',
    'user': 'icomp',
    'password': 'icomp123',
}

conn = psycopg2.connect(**config)
cur = conn.cursor()

# Configurar rich
from rich.table import Table
from rich.console import Console
from rich.syntax import Syntax
from rich.panel import Panel

console = Console()

---
## Tarefa 11
**Preparação e Verificação do Ambiente**

a) Execute o script movie.sql em movies para criar as tabelas e índices e carregar os dados necessários às próximas atividades

b) Verifique no catálogo do banco de dados os seguintes metadados sobre os índices associados às tabelas e apresente-os no relatório: Nome do índice, nome da tabela, altura, número máximo de chaves por bloco, número médio de chaves por bloco, número de blocos folha, número de médio de blocos folha por chave, número médio de blocos de dados por chave, número de linhas e número de chaves distintas.



### O que entregar
**Relatório com os resultados da verificação**

#### **A)** Execute o script `movie.sql` em movies para criar as tabelas e índices e carregar os dados necessários às próximas atividades

In [2]:
with open('movie.sql', 'r') as f:
    sql_script = f.read()
    cur.execute(sql_script)

# Checar se a tabela foi criada e os dados foram inseridos
cur.execute("""SELECT COUNT(*) FROM movie;""")
cur.fetchall()

[(1844,)]

#### **B)** Verifique no catálogo do banco de dados os seguintes metadados sobre os índices associados às tabelas e apresente-os no relatório: Nome do índice, nome da tabela, altura, número máximo de chaves por bloco, número médio de chaves por bloco, número de blocos folha, número de médio de blocos folha por chave, número médio de blocos de dados por chave, número de linhas e número de chaves distintas. 

Os metadados que são precisos para analisar índices da tabela `movie` são obtidos a partir dos seguintes catálogos:
- `pg_class`: fornece informações sobre tabelas e índices e armazena estatísticas básicas
- `pg_index`: relaciona cada índice à tabela correspondente
- `pg_attribute`: lista as colunas de tabelas e índices
- `pg_namespace`: define os esquemas do banco
- `pg_stat_all_indexes`: dá estatístiacs agregadas de uso dos índices

Algumas métricas específicas de organização física (altura da árvore, blocos folha, densidade de chaves etc.) só podem ser extraídas com a função `pgstatindex()`, fornecida pela extensão `pgstattuple`. É preciso que essa extensão esteja instalada e habilitada no BD. 

In [3]:
cur.execute("CREATE EXTENSION IF NOT EXISTS pgstattuple;")
conn.commit()

Identificamos todos os índices associados à tabela.  
Isso vem exclusivamente dos catálogos:

- `pg_class` para nomes de índices e tabelas  
- `pg_index` para relacionamentos  
- `pg_namespace` para filtrar o schema público

In [4]:
cur.execute("""
SELECT
    idx.relname AS index_name,
    tbl.relname AS table_name,
    pg_get_indexdef(i.indexrelid) AS index_def
FROM pg_index i
JOIN pg_class idx ON idx.oid = i.indexrelid
JOIN pg_class tbl ON tbl.oid = i.indrelid
JOIN pg_namespace n ON n.oid = tbl.relnamespace
WHERE n.nspname = 'public'
  AND tbl.relname = 'movie';
""")

indexes = cur.fetchall()

table = Table(title="Índices da tabela `movie`", title_style="bold bright_white")
table.add_column("Índice", header_style="bold bright_cyan")
table.add_column("Tabela", header_style="bold bright_cyan")
table.add_column("Definição", header_style="bold bright_cyan", overflow="fold")

for index_name, table_name, index_def in indexes:
    syntax = Syntax(index_def, "sql", theme="dracula", word_wrap=True)
    table.add_row(index_name, table_name, syntax)

console.print(table)

                                Índices da tabela `movie`                                
┏━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Índice      ┃ Tabela ┃ Definição                                                      ┃
┡━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ movie_key   │ movie  │ CREATE UNIQUE INDEX movie_key ON public.movie USING btree (id) │
│ movie_title │ movie  │ CREATE INDEX movie_title ON public.movie USING btree (title)   │
│ movie_votes │ movie  │ CREATE INDEX movie_votes ON public.movie USING btree (votes)   │
└─────────────┴────────┴────────────────────────────────────────────────────────────────┘

**Estatísticas internas dos índices (`pgstatindex`)**

A extensão `pgstattuple` fornece a função `pgstatindex()`, que nos revela
as propriedades internas da árvore B-Tree:

- `tree_level`: altura do índice  
- `leaf_pages`: quantidade de páginas folha  
- `avg_leaf_density`: densidade média de chaves nas folhas  
- `internal_pages`: páginas internas  
- `index_size`: tamanho total em bytes  
- `leaf_fragmentation`: fragmentação interna  

In [5]:
stats_per_index = {}

for idx, tbl, _ in indexes:
    cur.execute("SELECT * FROM pgstatindex(%s::regclass);", (idx,))
    row = cur.fetchone()
    cols = [d[0] for d in cur.description]
    stats_per_index[idx] = dict(zip(cols, row))

# mostrar estatísticas individualmente
for idx, stats in stats_per_index.items():
    t = Table(
        title=f"Estatísticas internas do índice: {idx}",
        title_style="bold bright_white"
    )
    t.add_column("Campo", header_style="bold bright_cyan")
    t.add_column("Valor", header_style="bold bright_cyan")

    for k, v in stats.items():
        t.add_row(k, str(v))

    console.print(t)

   Estatísticas internas do   
      índice: movie_key       
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Campo              ┃ Valor ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ version            │ 4     │
│ tree_level         │ 1     │
│ index_size         │ 57344 │
│ root_block_no      │ 3     │
│ internal_pages     │ 1     │
│ leaf_pages         │ 5     │
│ empty_pages        │ 0     │
│ deleted_pages      │ 0     │
│ avg_leaf_density   │ 90.73 │
│ leaf_fragmentation │ 0.0   │
└────────────────────┴───────┘

   Estatísticas internas do   
     índice: movie_title      
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Campo              ┃ Valor ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ version            │ 4     │
│ tree_level         │ 1     │
│ index_size         │ 98304 │
│ root_block_no      │ 3     │
│ internal_pages     │ 1     │
│ leaf_pages         │ 10    │
│ empty_pages        │ 0     │
│ deleted_pages      │ 0     │
│ avg_leaf_density   │ 72.33 │
│ leaf_fragmentation │ 40.0  │
└────────────────────┴───────┘

   Estatísticas internas do   
     índice: movie_votes      
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Campo              ┃ Valor ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ version            │ 4     │
│ tree_level         │ 1     │
│ index_size         │ 90112 │
│ root_block_no      │ 3     │
│ internal_pages     │ 1     │
│ leaf_pages         │ 9     │
│ empty_pages        │ 0     │
│ deleted_pages      │ 0     │
│ avg_leaf_density   │ 47.81 │
│ leaf_fragmentation │ 77.78 │
└────────────────────┴───────┘

A tabela abaixo reúne todas as informações solicitadas no enunciado:

- Nome do índice  
- Nome da tabela  
- Altura (tree_level)  
- Número máximo de chaves por bloco (estimado para PG ≥ 12)  
- Número médio de chaves por bloco (avg_leaf_density)  
- Número de blocos folha  
- Número médio de blocos folha por chave  
- Número médio de blocos de dados por chave (estimado)  
- Número de linhas  
- Número de chaves distintas  

As estimativas para `max_keys_per_block` e `avg_data_pages_per_key` são necessárias
porque o PostgreSQL moderno não expõe mais esses campos diretamente, mas podem
ser derivadas com base nos conceitos originais da função `pgstatindex()` e no tamanho
real da tabela.

In [6]:
cur.execute("SHOW block_size;")
BLOCK_SIZE = int(cur.fetchone()[0])
print(f"Tamanho do bloco: {BLOCK_SIZE} bytes")

Tamanho do bloco: 8192 bytes


In [7]:
rows = []

for idx_name, tbl_name, idx_def in indexes:

    stats = stats_per_index[idx_name]

    # total de linhas da tabela
    cur.execute(f"SELECT COUNT(*) FROM {tbl_name};")
    total_rows = cur.fetchone()[0]

    # colunas do índice
    cur.execute("""
        SELECT a.attname
        FROM pg_attribute a
        JOIN pg_index i ON a.attrelid = i.indrelid
                       AND a.attnum = ANY(i.indkey)
        JOIN pg_class c ON c.oid = i.indexrelid
        WHERE c.relname = %s;
    """, (idx_name,))
    cols = [r[0] for r in cur.fetchall()]
    col_expr = ", ".join(cols)

    # número de chaves distintas
    cur.execute(f"SELECT COUNT(DISTINCT ({col_expr})) FROM {tbl_name};")
    distinct = cur.fetchone()[0]

    # páginas folha
    leaf_pages = stats["leaf_pages"]

    # média de blocos folha por chave (exato)
    avg_leaf_pages_per_key = (
        leaf_pages / distinct if distinct > 0 else None
    )

    # estimativa para max_leaf_keys
    # capacidade máxima ≈ densidade * espaço útil
    max_leaf_keys_estimated = int((stats["avg_leaf_density"] / 100) * 8192)

    # estimativa para média de páginas de dados por chave
    cur.execute(f"SELECT pg_relation_size('{tbl_name}'::regclass);")
    table_size_bytes = cur.fetchone()[0]
    heap_pages = table_size_bytes / BLOCK_SIZE

    avg_data_pages_per_key_estimated = (
        heap_pages / distinct if distinct > 0 else None
    )

    rows.append({
        "index_name": idx_name,
        "table_name": tbl_name,
        "height": stats["tree_level"],
        "max_keys_per_block": max_leaf_keys_estimated,
        "avg_keys_per_block": stats["avg_leaf_density"],
        "leaf_pages": leaf_pages,
        "avg_leaf_pages_per_key": avg_leaf_pages_per_key,
        "avg_data_pages_per_key": avg_data_pages_per_key_estimated,
        "total_rows": total_rows,
        "distinct_keys": distinct
    })

In [8]:
table = Table(
    title="Metadados Consolidados dos Índices da Tabela `movie`",
    title_style="bold bright_white"
)

# colunas com estilo claro e consistência
columns = [
    "Índice",
    "Tabela",
    "Altura",
    "Máx. chaves/bloco (est.)",
    "Méd. chaves/bloco",
    "Blocos folha",
    "Méd. bloco folha/chave",
    "Méd. bloco dados/chave (est.)",
    "Linhas",
    "Chaves distintas"
]

for col in columns:
    table.add_column(col, header_style="bold bright_cyan", overflow="fold")

# preenchimento da tabela
for r in rows:
    table.add_row(
        r["index_name"],
        r["table_name"],
        str(r["height"]),
        f"{r['max_keys_per_block']:.2f}",
        f"{r['avg_keys_per_block']:.2f}",
        str(r["leaf_pages"]),
        f"{r['avg_leaf_pages_per_key']:.4f}",
        f"{r['avg_data_pages_per_key']:.4f}",
        str(r["total_rows"]),
        str(r["distinct_keys"])
    )

console.print(table)

                               Metadados Consolidados dos Índices da Tabela `movie`                                
┏━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┓
┃           ┃        ┃        ┃ Máx.      ┃            ┃           ┃            ┃ Méd.      ┃        ┃            ┃
┃           ┃        ┃        ┃ chaves/bl ┃ Méd.       ┃           ┃ Méd. bloco ┃ bloco     ┃        ┃            ┃
┃           ┃        ┃        ┃ oco       ┃ chaves/blo ┃ Blocos    ┃ folha/chav ┃ dados/cha ┃        ┃ Chaves     ┃
┃ Índice    ┃ Tabela ┃ Altura ┃ (est.)    ┃ co         ┃ folha     ┃ e          ┃ ve (est.) ┃ Linhas ┃ distintas  ┃
┡━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━┩
│ movie_key │ movie  │ 1      │ 7432.00   │ 90.73      │ 5         │ 0.0027     │ 0.0081    │ 1844   │ 1844       │
│ movie_tit │ movie  │ 1      │ 5925.00   │ 72.33      │ 10        │ 0.0055     │ 0.0082    │ 1844   │ 1833       │
│ le        │        │        │           │            │           │            │           │        │            │
│ movie_vot │ movie  │ 1      │ 3916.00   │ 47.81      │ 9         │ 0.0061     │ 0.0101    │ 1844   │ 1481       │
│ es        │        │        │           │            │           │            │           │        │            │
└───────────┴────────┴────────┴───────────┴────────────┴───────────┴────────────┴───────────┴────────┴────────────┘

---
## Tarefa 12
**Consultas por intervalo e índices secundários**

a) Escreva uma consulta em SQL sobre o atributo VOTES da tabela MOVIE que recupera um número pequeno de tuplas (<10 tuplas); Execute o comando `EXPLAIN ANALYZE` sobre esta consulta e apresente os resultados

b) Escreva uma consulta em SQL sobre o atributo VOTES da tabela MOVIE que recupera um número grande de tuplas (>80% das tuplas). Execute o comando `EXPLAIN ANALYZE` sobre esta consulta e apresente os resultados

c) Explique porque o índice sobre VOTES não é sempre usado nas consultas sobre este atributo


### O que entregar
**Relatório com as respostas das questões**


#### **A)** Escreva uma consulta em SQL sobre o atributo VOTES da tabela MOVIE que recupera um número pequeno de tuplas (<10 tuplas); Execute o comando `EXPLAIN ANALYZE` sobre esta consulta e apresente os resultados

Pegar 10 tuplas distintas a partir do 50º registro ordenado por votos para ter uma noção de poucas tuplas.

In [18]:
cur.execute("""SELECT DISTINCT votes FROM movie ORDER BY votes LIMIT 10 OFFSET 50;""")
cur.fetchall()

[(802,),
 (803,),
 (805,),
 (806,),
 (807,),
 (808,),
 (809,),
 (811,),
 (812,),
 (813,)]

Assim, escolhemos um itervalo restrito entre 800 e 810 votos:
```postgresql
SELECT * FROM movie WHERE votes BETWEEN 800 AND 810;
```

In [21]:
query_small = """
EXPLAIN ANALYZE
SELECT *
FROM movie
WHERE votes BETWEEN 900000 AND 901000;
"""

cur.execute(query_small)
plan_small = "\n".join(row[0] for row in cur.fetchall())

console.print(Panel(plan_small, title="Plano — Consulta Seletiva (<10 tuplas)"))

╭──────────────────────────────────── Plano — Consulta Seletiva (<10 tuplas) ─────────────────────────────────────╮
│ Index Scan using movie_votes on movie  (cost=0.28..8.30 rows=1 width=30) (actual time=0.004..0.005 rows=0       │
│ loops=1)                                                                                                        │
│   Index Cond: ((votes >= 900000) AND (votes <= 901000))                                                         │
│ Planning Time: 0.165 ms                                                                                         │
│ Execution Time: 0.025 ms                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Dessa forma, conseguimos ver que o plano de execução utiliza o índice sobre `votes` para buscar as tuplas desejadas.

#### **B)** Escreva uma consulta em SQL sobre o atributo VOTES da tabela MOVIE que recupera um número grande de tuplas (>80% das tuplas). Execute o comando `EXPLAIN ANALYZE` sobre esta consulta e apresente os resultados

In [22]:
query_big = """
EXPLAIN ANALYZE
SELECT *
FROM movie
WHERE votes > 100;"""

cur.execute(query_big)
plan_big = "\n".join(row[0] for row in cur.fetchall())

console.print(Panel(plan_big, title="Plano — Consulta Não Seletiva (>80% tuplas)"))

╭────────────────────────────────── Plano — Consulta Não Seletiva (>80% tuplas) ──────────────────────────────────╮
│ Seq Scan on movie  (cost=0.00..38.05 rows=1844 width=30) (actual time=0.013..0.363 rows=1844 loops=1)           │
│   Filter: (votes > 100)                                                                                         │
│ Planning Time: 0.249 ms                                                                                         │
│ Execution Time: 0.438 ms                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Em consultas não seletivas, o otimizador opta por não usar o índice, realizando um **seq scan** na tabela.

#### **C)** Explique porque o índice sobre VOTES não é sempre usado nas consultas sobre este atributo

O índice secundário sobre `votes` não é sempre utilizado devido às regras de custo do otimizador do SGBD.

Para retornar poucas tuplas (< 5–10%), é mais barato caminhar pela B-tree outraçar um intervalo nas folhas da árvore e buscar as tuplas correspondentes no heap.

Quando a consulta retorna muitas tuplas (> 30–40%), o custo muda. O SGBD teria que:
- seguir muitas folhas da B-tree  
- fazer milhares de acessos aleatórios ao heap  

Essa forma acaba sendo mais cara que uma varredura, que lê as páginas da tabela sequencialmente.